In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    preprocessor_obj_file_path: str
    train_data_path: str
    model_name: str
    n_estimators: float
    max_depth: float
    min_samples_split: int
    min_samples_leaf: int
    target_column: str

In [6]:
from Laptop_Price_Prediction_System.constants import *
from Laptop_Price_Prediction_System.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.Random_Forest
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            preprocessor_obj_file_path=config.preprocessor_obj_file_path,
            train_data_path=config.train_data_path,
            model_name=config.model_name,
            n_estimators=params.n_estimators,
            max_depth=params.max_depth,
            min_samples_split=params.min_samples_split,
            min_samples_leaf=params.min_samples_leaf,
            target_column=schema.name
        )

        return model_trainer_config

In [ ]:
import pandas as pd
import os
from Laptop_Price_Prediction_System import logger
from sklearn.ensemble import RandomForestRegressor
import joblib

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
    
    def train(self):
        logger.info("Loading Training Data")
        train_data = pd.read_csv(self.config.train_data_path)

        logger.info("Loading Preprocessor")
        preprocessor = joblib.load(self.config.preprocessor_obj_file_path)

        train_X = train_data.drop([self.config.target_column], axis=1)
        train_y = train_data[self.config.target_column]

        X_processed = preprocessor.transform(train_X)

        logger.info("Training Model")
        model = RandomForestRegressor(n_estimators=self.config.n_estimators, max_depth=self.config.max_depth, min_samples_split=self.config.min_samples_split, min_samples_leaf=self.config.min_samples_leaf)
        model.fit(X_processed, train_y)
        
        logger.info("Saving Model")
        joblib.dump(model, os.path.join(self.config.root_dir, self.config.model_name))

In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-07-10 08:48:46,234 : INFO : common : Loaded file: config/config.yaml]
[2026-07-10 08:48:46,237 : INFO : common : Loaded file: params.yaml]
[2026-07-10 08:48:46,240 : INFO : common : Loaded file: schema.yaml]
[2026-07-10 08:48:46,241 : INFO : common : Created directory: artifacts]
[2026-07-10 08:48:46,243 : INFO : common : Created directory: artifacts/model_trainer]
